# Background

This project aims to replicate the "Add a column from examples" feature found in Power BI Desktop. In Power BI, this feature allows users to add a new column by specifying examples of the desired transformation, making data processing intuitive and straightforward. However, the code provided here focuses primarily on recognizing text transformation patterns rather than directly performing the transformations.

Project Objectives
Pattern Recognition: The code identifies transformation patterns based on examples provided by the user.
Custom Implementation: Users can implement their transformations and share their modifications.

Features
This project includes several text transformation patterns that users can recognize and implement:
```
Combine: Combines literal strings or entire column values.
Replace: Replaces a specific substring with another string.
Length: Computes the length of a string.
Extract: Extracts a substring starting from a specific position with a defined length.
First Characters: Extracts the first n characters of a string.
Last Characters: Extracts the last n characters of a string.
Range: Extracts a range of characters within specified start and end indices.
Text before Delimiter: Extracts the text before a specified delimiter.
Text after Delimiter: Extracts the text after a specified delimiter.
Text between Delimiters: Extracts text between two delimiters.
Remove Characters: Removes specific characters from a string.
Keep Characters: Retains only specified characters in a string.
```

## Invitation to Collaborate
All readers are encouraged to experiment with this pattern recognition feature and expand on it to implement the actual transformations. Feel free to share the resulting code with the community and contribute to this open-source project.

# Import

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import load_model
from collections import Counter

# Load Data

In [ ]:
df = pd.read_csv('/kaggle/input/dataset/text_transformation_examples.csv', delimiter='|', quotechar='"')

# Data Cleaning

In [ ]:
df.dropna(inplace=True)  # Remove rows with missing values

# Feature and Label Selection

In [ ]:
X = df[['Input', 'Output']].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
y = df['Transformation']

# Text Tokenization and Padding

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=100)  # Assume a maximum sequence length of 100

# Label Encoding and One-Hot Encoding

In [ ]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)

# Print Shapes for Debugging

In [ ]:
print("Shape of X_pad:", X_pad.shape)
print("Shape of y_categorical:", y_categorical.shape)

# Split Data into Training and Testing Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pad, y_categorical, test_size=0.2, random_state=42)
train_labels = set(y_train.argmax(axis=1))
test_labels = set(y_test.argmax(axis=1))
unseen_labels = test_labels - train_labels
print("Unseen labels in test data:", unseen_labels)

# Model Parameters

In [ ]:
vocab_size = len(tokenizer.word_index) + 1  # Total number of unique words
embedding_dim = 50  # Dimension of the embedding vector
num_classes = y_categorical.shape[1]  # Number of classes

# Build Model

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=100))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dense(num_classes, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model

In [ ]:
model.fit(X_train, y_train, epochs=6, batch_size=32, validation_data=(X_test, y_test))

# Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Save and Load Model

In [ ]:
model.save("transform_model.h5")
print("Model saved successfully.")
loaded_model = load_model("transform_model.h5")
print("Model loaded successfully.")

# Prediction with Loaded Model

In [ ]:
predictions_loaded_model = loaded_model.predict(X_pad)
predicted_class_loaded_model = encoder.inverse_transform(predictions_loaded_model.argmax(axis=-1))
#print(f"Predicted Transformation with loaded model: {predicted_class_loaded_model[0]}")

# Function to Predict Transformation Pattern

In [ ]:
def predict_transformation(df, column_name, tokenizer, encoder, model, examples):
    # Create a copy of the original DataFrame to make modifications
    predicted_df = df.copy()

    # Replace example values in the specified column
    num_examples = len(examples)
    predicted_df.loc[:num_examples-1, column_name] = examples

    # Tokenize the examples
    examples_seq = tokenizer.texts_to_sequences(predicted_df[column_name][:num_examples].astype(str))
    examples_pad = pad_sequences(examples_seq, maxlen=100)

    # Predict using the model
    predictions = model.predict(examples_pad)

    # Convert predictions into transformation classes
    predicted_classes = encoder.inverse_transform(predictions.argmax(axis=-1))

    # Aggregate the most common pattern
    most_common_pattern = Counter(predicted_classes).most_common(1)[0][0]

    return most_common_pattern

# Test the Pattern Prediction Function

In [ ]:
building_permits_df = pd.read_csv('/kaggle/input/dataset/Building_Permits.csv', low_memory=False)
building_permits_df['Permit Number'] = building_permits_df['Permit Number'].astype(str)
loaded_model = load_model("transform_model.h5")
new_data = ['19', '46', '09'] # last 2 characters of the string as a test

# Make Prediction

In [ ]:
predicted_pattern = predict_transformation(building_permits_df, 'Permit Number', tokenizer, encoder, loaded_model, new_data)
print(f"Identified Pattern: {predicted_pattern}")

First 4 rows of the orginal column:
``` 
0         201505065519
1         201604195146
2         201605278609
3         201611072166
4         201611283529
...
``` 


# Script for generating the dataset for training

In [ ]:
import random
import string
import pandas as pd
import csv

def random_string(length=10):
    letters = string.ascii_letters + string.digits + '#;/,.'
    return ''.join(random.choice(letters) for i in range(length))

def combine(s1, s2):
    return f"{s1}{s2}", 'Combine'

def replace(input_string):
    if input_string and len(input_string) > 1:
        old_char = random.choice([char for char in input_string if char.isalnum() or char in '#;/,.'])
        new_char = random.choice(string.ascii_letters + string.digits + '#;/,.')
        result = input_string.replace(old_char, new_char, 1)
        if result != input_string:
            return result, 'Replace'
    return input_string, 'Replace (no action)'

def string_length(input_string):
    if input_string:
        return len(input_string), 'Length'
    return 0, 'Length (no action)'

def extract(input_string, start, length):
    if input_string and 0 <= start < len(input_string) and start + length <= len(input_string):
        return input_string[start:start+length], 'Extract'
    return input_string, 'Extract (no action)'

def first_characters(input_string, n):
    if input_string and 0 < n <= len(input_string):
        return input_string[:n], 'First Characters'
    return input_string, 'First Characters (no action)'

def last_characters(input_string, n):
    if input_string and 0 < n <= len(input_string):
        return input_string[-n:], 'Last Characters'
    return input_string, 'Last Characters (no action)'

def range_characters(input_string, start, end):
    if end > start and end <= len(input_string):
        return input_string[start:end], 'Range'
    return input_string, 'Range (no action)'

def text_before_delimiter(input_string, delimiter):
    parts = input_string.split(delimiter)
    if len(parts) > 1:
        return parts[0], 'Text before Delimiter'
    return input_string, 'Text before Delimiter (no action)'

def text_after_delimiter(input_string, delimiter):
    parts = input_string.split(delimiter)
    if len(parts) > 1:
        return parts[1], 'Text after Delimiter'
    return input_string, 'Text after Delimiter (no action)'

def text_between_delimiters(input_string, start_delim, end_delim):
    start = input_string.find(start_delim)
    if start != -1:
        start += len(start_delim)
        end = input_string.find(end_delim, start)
        if end != -1:
            return input_string[start:end], 'Text between Delimiters'
    return input_string, 'Text between Delimiters (no action)'

def remove_characters(input_string, chars_to_remove):
    result = "".join(char for char in input_string if char not in chars_to_remove)
    if result != input_string and result:
        return result, 'Remove Characters'
    return input_string, 'Remove Characters (no action)'

def keep_characters(input_string, chars_to_keep):
    result = "".join(char for char in input_string if char in chars_to_keep)
    if result:
        return result, 'Keep Characters'
    return input_string, 'Keep Characters (no action)'

def generate_data(num_samples=200000):
    data = []
    transformations = [
        'Combine', 'Replace', 'Length', 'Extract', 'First Characters', 'Last Characters',
        'Text before Delimiter', 'Text after Delimiter', 'Text between Delimiters',
        'Remove Characters', 'Keep Characters'
    ]
    delimiters = [',', ';', '/']
    for _ in range(num_samples):
        original = random_string(random.randint(5, 15))
        transformation_type = random.choice(transformations)
        result, trans_description = '', ''

        if transformation_type == 'Combine':
            result, trans_description = combine(original, random_string(random.randint(1, 10)))
        elif transformation_type == 'Replace':
            result, trans_description = replace(original)
        elif transformation_type == 'Length':
            result, trans_description = string_length(original)
        elif transformation_type in ['First Characters', 'Last Characters', 'Extract']:
            n = random.randint(1, 10)
            if transformation_type == 'Extract':
                start = random.randint(0, len(original) - 1)
                result, trans_description = extract(original, start, n)
            elif transformation_type == 'First Characters':
                result, trans_description = first_characters(original, n)
            elif transformation_type == 'Last Characters':
                result, trans_description = last_characters(original, n)
        elif transformation_type in ['Text before Delimiter', 'Text after Delimiter', 'Text between Delimiters']:
            delimiter = random.choice(delimiters)
            if transformation_type == 'Text before Delimiter':
                result, trans_description = text_before_delimiter(original, delimiter)
            elif transformation_type == 'Text after Delimiter':
                result, trans_description = text_after_delimiter(original, delimiter)
            else:
                end_delim = random.choice(delimiters)
                result, trans_description = text_between_delimiters(original, delimiter, end_delim)
        elif transformation_type == 'Remove Characters':
            chars_to_remove = random_string(random.randint(1, 5))
            result, trans_description = remove_characters(original, chars_to_remove)
        elif transformation_type == 'Keep Characters':
            chars_to_keep = random_string(random.randint(1, 5))
            result, trans_description = keep_characters(original, chars_to_keep)

        if 'no action' not in trans_description:
            data.append((original, str(result), trans_description))

    return data

def save_data_to_csv(data, filename='text_transformation_examples.csv'):
    df = pd.DataFrame(data, columns=['Input', 'Output', 'Transformation'])
    df.to_csv(
        filename, 
        index=False, 
        sep='|', 
        quoting=csv.QUOTE_NONE, 
        escapechar='\\'
    )

data_examples = generate_data()
save_data_to_csv(data_examples)